In [ ]:
import pandas as pd
import numpy as np
import sklearn
import os
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold

In [ ]:
!unzip test.zip -d /content/ #압축풀기

Archive:  test.zip
   creating: /content/test/
  inflating: /content/__MACOSX/._test  
   creating: /content/test/153390/
  inflating: /content/__MACOSX/test/._153390  
   creating: /content/test/153364/
  inflating: /content/__MACOSX/test/._153364  
   creating: /content/test/153363/
  inflating: /content/__MACOSX/test/._153363  
   creating: /content/test/153365/
  inflating: /content/__MACOSX/test/._153365  
   creating: /content/test/153391/
  inflating: /content/__MACOSX/test/._153391  
  inflating: /content/test/.DS_Store  
  inflating: /content/__MACOSX/test/._.DS_Store  
   creating: /content/test/153378/
  inflating: /content/__MACOSX/test/._153378  
   creating: /content/test/153382/
  inflating: /content/__MACOSX/test/._153382  
   creating: /content/test/153376/
  inflating: /content/__MACOSX/test/._153376  
   creating: /content/test/153371/
  inflating: /content/__MACOSX/test/._153371  
   creating: /content/test/153385/
  inflating: /content/__MACOSX/test/._153385  
   c

In [ ]:
#파일 설정
path_train="train.csv"
path_test_index="test.csv"
path_match_info="match_info.csv"
path_sample_sub="/content/sample_submission.csv"


In [ ]:
train=pd.read_csv(path_train)
test=pd.read_csv(path_test_index)
match_info=pd.read_csv(path_match_info)
sample_sub=pd.read_csv(path_sample_sub)

In [ ]:
test.head()

,game_id,game_episode,path
0,153363,153363_1,./test/153363/153363_1.csv
1,153363,153363_2,./test/153363/153363_2.csv
2,153363,153363_6,./test/153363/153363_6.csv
3,153363,153363_7,./test/153363/153363_7.csv
4,153363,153363_8,./test/153363/153363_8.csv


In [ ]:
test_events_list = []

for _, row in test.iterrows():
    full_path = row['path'].replace('./', '/content/') # row['path'] 경로를 코랩 경로로 바꿈
    df_ep = pd.read_csv(full_path)
    test_events_list.append(df_ep)

test_events = pd.concat(test_events_list, ignore_index=True)
test_events.columns

Index(['game_id', 'period_id', 'episode_id', 'time_seconds', 'team_id',
       'player_id', 'action_id', 'type_name', 'result_name', 'start_x',
       'start_y', 'end_x', 'end_y', 'is_home', 'game_episode'],
      dtype='object')

In [ ]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 356721 entries, 0 to 356720
Data columns (total 15 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   game_id       356721 non-null  int64  
 1   period_id     356721 non-null  int64  
 2   episode_id    356721 non-null  int64  
 3   time_seconds  356721 non-null  float64
 4   team_id       356721 non-null  int64  
 5   player_id     356721 non-null  int64  
 6   action_id     356721 non-null  int64  
 7   type_name     356721 non-null  object 
 8   result_name   216467 non-null  object 
 9   start_x       356721 non-null  float64
 10  start_y       356721 non-null  float64
 11  end_x         356721 non-null  float64
 12  end_y         356721 non-null  float64
 13  is_home       356721 non-null  bool   
 14  game_episode  356721 non-null  object 
dtypes: bool(1), float64(5), int64(6), object(3)
memory usage: 38.4+ MB


In [ ]:
train['is_train']=1
test_events['is_train']=0

events=pd.concat([train,test_events],ignore_index=True)
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 409831 entries, 0 to 409830
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   game_id       409831 non-null  int64  
 1   period_id     409831 non-null  int64  
 2   episode_id    409831 non-null  int64  
 3   time_seconds  409831 non-null  float64
 4   team_id       409831 non-null  int64  
 5   player_id     409831 non-null  int64  
 6   action_id     409831 non-null  int64  
 7   type_name     409831 non-null  object 
 8   result_name   248448 non-null  object 
 9   start_x       409831 non-null  float64
 10  start_y       409831 non-null  float64
 11  end_x         407417 non-null  float64
 12  end_y         407417 non-null  float64
 13  is_home       409831 non-null  bool   
 14  game_episode  409831 non-null  object 
 15  is_train      409831 non-null  int64  
dtypes: bool(1), float64(5), int64(7), object(3)
memory usage: 47.3+ MB


In [ ]:
#정렬 및 episode 내 인덱스
events=events.sort_values(['game_episode','time_seconds','action_id']) #정렬


events['event_idx']=events.groupby('game_episode').cumcount() #episode별로 묶고 0부터 시작하는 누적 카운트 컬럼 생성
events['n_events']=events.groupby('game_episode')['event_idx'].transform('max')+1 #event_idx의 최댓값은 마지막 번호에 +1
events['ep_idx_norm']=events['event_idx']/(events['n_events']-1).clip(lower=1) #ep_idx_norm 이벤트가 얼마나 진행됐는지 확인하는 비율

events.head(5)

,game_id,period_id,episode_id,time_seconds,team_id,player_id,action_id,type_name,result_name,start_x,start_y,end_x,end_y,is_home,game_episode,is_train,event_idx,n_events,ep_idx_norm
0,126283,1,1,0.667,2354,344559,0,Pass,Successful,52.418205,33.485444,31.322445,38.274752,True,126283_1,1,0,49,0.000000
1,126283,1,1,3.667,2354,250036,2,Pass,Successful,32.013240,38.100808,37.371285,30.632980,True,126283_1,1,1,49,0.020833
2,126283,1,1,4.968,2354,500145,4,Carry,NaN,37.371285,30.632980,38.391570,24.613144,True,126283_1,1,2,49,0.041667
3,126283,1,1,8.200,2354,500145,5,Pass,Successful,38.391570,24.613144,34.573350,5.545468,True,126283_1,1,3,49,0.062500
4,126283,1,1,11.633,2354,142106,7,Pass,Successful,34.578705,6.058256,21.274470,18.437112,True,126283_1,1,4,49,0.083333


In [ ]:
#시간/공간 feature

## episode별로 묶어서 각 이벤트 사이의 시간 차이
events['prev_time']=events.groupby('game_episode')['time_seconds'].shift(1) #각 이벤트의 이전 값을 pre_time컬럼으로 생성
events['dt']=events['time_seconds']-events['prev_time'] #dt=현재시간-이전시간
events['dt']=events['dt'].fillna(0.0) #첫 이벤트의 값은 0으로 채우기

events.head(5)

## 이동량/거리
events['dx']=events['end_x']-events['start_x']
events['dy']=events['end_y']-events['start_y']
events['dist']=np.sqrt(events['dx']**2+events['dy']**2)

## 속도(dt=0)
events['speed']=events['dist']/events['dt'].replace(0,1e-3) #0을 아주 작은 값인 0.001로 바꾼다

## zone/lane(필요시 범위 조정)
events['x_zone']=(events['start_x']/(105/7)).astype(int).clip(0,6)
#축구장 가로 길이를 7개의 구역으로 나눔, 105/7=약 15, ex) start_x=42 이면 zone.astype(int)=2(2.8)
#zone은 반드시 0~6 사이여야함
events['lane']=pd.cut(
    events['start_y'],
    bins=[0,68/3,2*68/3,68], #세로 길이 68m를 3개의 레인으로 구분
    labels=[0,1,2],
    include_lowest=True).astype(int) #필드 위치를 구간으로 나누는 작업

#start_y값이 어느 구간에 있는지에 따라 lane 3개로 분류

#따라서 현재 zone+lane 조합으로 grid(7X3=21칸)임

,game_id,period_id,episode_id,time_seconds,team_id,player_id,action_id,type_name,result_name,start_x,...,end_x,end_y,is_home,game_episode,is_train,event_idx,n_events,ep_idx_norm,prev_time,dt
0,126283,1,1,0.667,2354,344559,0,Pass,Successful,52.418205,...,31.322445,38.274752,True,126283_1,1,0,49,0.000000,NaN,0.000
1,126283,1,1,3.667,2354,250036,2,Pass,Successful,32.013240,...,37.371285,30.632980,True,126283_1,1,1,49,0.020833,0.667,3.000
2,126283,1,1,4.968,2354,500145,4,Carry,NaN,37.371285,...,38.391570,24.613144,True,126283_1,1,2,49,0.041667,3.667,1.301
3,126283,1,1,8.200,2354,500145,5,Pass,Successful,38.391570,...,34.573350,5.545468,True,126283_1,1,3,49,0.062500,4.968,3.232
4,126283,1,1,11.633,2354,142106,7,Pass,Successful,34.578705,...,21.274470,18.437112,True,126283_1,1,4,49,0.083333,8.200,3.433


In [ ]:
#라벨 변경 및 episode-level 메타(train 전용)
train_events=events[events['is_train']==1].copy()
train_events.head()

last_events=(
    train_events.groupby('game_episode',as_index=False)
    .tail(1).copy()
) #game_episode 그룹별 마지막 이벤트만 추출(game_episode의 마지막 행만 추출)


print(last_events.head(5))
labels=last_events[['game_episode','end_x','end_y']].rename(
    columns={'end_x':'target_x','end_y':'target_y'}
)
print("\n")
print(labels.head(5))

#episode-level 메타(마지막 이벤트 기준)
ep_meta=last_events[['game_episode','game_id','team_id','is_home','period_id','time_seconds']].copy()
ep_meta=ep_meta.rename(columns={'team_id':'final_team_id'})

#game_clock: 경기 누적 시간, 이때 하프타임은 경기 시간에 포함X (분 단위, 0~90+)
ep_meta['game_clock_min']=np.where(
    ep_meta['period_id']==1, #period_id=1일떄는 전반(45분) 2일때는 후반(45분) +
    ep_meta['time_seconds']/60.0, #전반 전체시간 분으로 바꿈
    45.0+ep_meta['time_seconds']/60.0 #후반 전체시간 45분 + 후반 시작 시간
    )

     game_id  period_id  episode_id  time_seconds  team_id  player_id  \
48    126283          1           1       124.367     2354     500146   
50    126283          1          10       695.473     4639     412018   
146   126283          1          11       894.237     4639      97086   
149   126283          1          12       940.933     2354      61979   
177   126283          1          13       997.233     4639     142258   

     action_id type_name   result_name     start_x  ...  game_episode  \
48          68      Pass    Successful  101.054205  ...      126283_1   
50         425      Pass    Successful  102.213510  ...     126283_10   
146        573      Pass    Successful   67.934265  ...     126283_11   
149        588      Pass  Unsuccessful   80.991120  ...     126283_12   
177        629      Pass  Unsuccessful   14.527695  ...     126283_13   

     is_train  event_idx  n_events ep_idx_norm  prev_time     dt  \
48          1         48        49         1.0    122.

In [ ]:
events.head(5)

,game_id,period_id,episode_id,time_seconds,team_id,player_id,action_id,type_name,result_name,start_x,...,game_episode,is_train,event_idx,n_events,ep_idx_norm,prev_time,dt,final_team_id_x,is_final_team,final_team_id_y
0,126283,1,1,0.667,2354,344559,0,Pass,Successful,52.418205,...,126283_1,1,0,49,0.000000,NaN,0.000,2354.0,1,2354.0
1,126283,1,1,3.667,2354,250036,2,Pass,Successful,32.013240,...,126283_1,1,1,49,0.020833,0.667,3.000,2354.0,1,2354.0
2,126283,1,1,4.968,2354,500145,4,Carry,NaN,37.371285,...,126283_1,1,2,49,0.041667,3.667,1.301,2354.0,1,2354.0
3,126283,1,1,8.200,2354,500145,5,Pass,Successful,38.391570,...,126283_1,1,3,49,0.062500,4.968,3.232,2354.0,1,2354.0
4,126283,1,1,11.633,2354,142106,7,Pass,Successful,34.578705,...,126283_1,1,4,49,0.083333,8.200,3.433,2354.0,1,2354.0


In [ ]:
#공격 팀 플래그(final_team VS 상대)

#final_team_id를 전체 events에 붙임
events=events.merge(
    ep_meta[['game_episode','final_team_id']],
    on='game_episode',
    how='left'
) # game_episode 기준으로 ep_meta테이블과 events테이블 연결 그래서 ep_meta의 정보 없으면 NaN임

events['is_final_team']=(events['team_id']==events['final_team_id']).astype(int)
#둘이 같아서 True는 1로 변환, False는 0으로 변환

events

,game_id,period_id,episode_id,time_seconds,team_id,player_id,action_id,type_name,result_name,start_x,...,is_train,event_idx,n_events,ep_idx_norm,prev_time,dt,final_team_id_x,is_final_team,final_team_id_y,final_team_id
0,126283,1,1,0.667,2354,344559,0,Pass,Successful,52.418205,...,1,0,49,0.000000,NaN,0.000,2354.0,1,2354.0,2354.0
1,126283,1,1,3.667,2354,250036,2,Pass,Successful,32.013240,...,1,1,49,0.020833,0.667,3.000,2354.0,1,2354.0,2354.0
2,126283,1,1,4.968,2354,500145,4,Carry,NaN,37.371285,...,1,2,49,0.041667,3.667,1.301,2354.0,1,2354.0,2354.0
3,126283,1,1,8.200,2354,500145,5,Pass,Successful,38.391570,...,1,3,49,0.062500,4.968,3.232,2354.0,1,2354.0,2354.0
4,126283,1,1,11.633,2354,142106,7,Pass,Successful,34.578705,...,1,4,49,0.083333,8.200,3.433,2354.0,1,2354.0,2354.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
409826,153392,2,99,2566.267,4641,145831,2489,Pass,Successful,46.779180,...,0,3,8,0.428571,2564.367,1.900,NaN,0,NaN,NaN
409827,153392,2,99,2567.734,4641,532197,2491,Carry,NaN,62.483715,...,0,4,8,0.571429,2566.267,1.467,NaN,0,NaN,NaN
409828,153392,2,99,2569.500,4657,500478,2492,Tackle,Unsuccessful,49.400085,...,0,5,8,0.714286,2567.734,1.766,NaN,0,NaN,NaN
409829,153392,2,99,2570.600,4641,532197,2493,Pass,Successful,50.215935,...,0,6,8,0.857143,2569.500,1.100,NaN,0,NaN,NaN


In [ ]:
# 입력용 events에서 마지막 이벤트 타깃 정보 가리기

#is_last 플래그
events['last_idx']=events.groupby('game_episode')['event_idx'].transform('max')
#game_episode 별로 event_idx의 최댓값=마지막 득점의 번호
events['is_last']=(events['event_idx']==events['last_idx']).astype(int)
#둘이 같아서 True는 1로 변환, False는 0으로 변환

#labels는 이미 뽑아놨으니 입력쪽에서만 end_x,end_y,dx,dy,dist,speed 지움
mask_last=events['is_last']==1 #같은 경우
for col in ['end_x','end_y','dx','dy','dist','speed']:
  events.loc[mask_last,col]=np.nan #마지막 득점의 다음 득점의 이동 정보는 존재할 수가 없어서 NaN으로 처리

events.head(5)

,game_id,period_id,episode_id,time_seconds,team_id,player_id,action_id,type_name,result_name,start_x,...,final_team_id_x,is_final_team,final_team_id_y,final_team_id,last_idx,is_last,dx,dy,dist,speed
0,126283,1,1,0.667,2354,344559,0,Pass,Successful,52.418205,...,2354.0,1,2354.0,2354.0,48,0,NaN,NaN,NaN,NaN
1,126283,1,1,3.667,2354,250036,2,Pass,Successful,32.013240,...,2354.0,1,2354.0,2354.0,48,0,NaN,NaN,NaN,NaN
2,126283,1,1,4.968,2354,500145,4,Carry,NaN,37.371285,...,2354.0,1,2354.0,2354.0,48,0,NaN,NaN,NaN,NaN
3,126283,1,1,8.200,2354,500145,5,Pass,Successful,38.391570,...,2354.0,1,2354.0,2354.0,48,0,NaN,NaN,NaN,NaN
4,126283,1,1,11.633,2354,142106,7,Pass,Successful,34.578705,...,2354.0,1,2354.0,2354.0,48,0,NaN,NaN,NaN,NaN


In [ ]:
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 409831 entries, 0 to 409830
Data columns (total 31 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   game_id          409831 non-null  int64  
 1   period_id        409831 non-null  int64  
 2   episode_id       409831 non-null  int64  
 3   time_seconds     409831 non-null  float64
 4   team_id          409831 non-null  int64  
 5   player_id        409831 non-null  int64  
 6   action_id        409831 non-null  int64  
 7   type_name        409831 non-null  object 
 8   result_name      248448 non-null  object 
 9   start_x          409831 non-null  float64
 10  start_y          409831 non-null  float64
 11  end_x            391982 non-null  float64
 12  end_y            391982 non-null  float64
 13  is_home          409831 non-null  bool   
 14  game_episode     409831 non-null  object 
 15  is_train         409831 non-null  int64  
 16  event_idx        409831 non-null  int6

In [ ]:
events['type_name'].value_counts()

,count
type_name,
Pass,204612
Carry,94037
Recovery,31703
Interception,12817
Duel,10154
Tackle,9435
Throw-In,7904
Clearance,7527
Intervention,7044


In [ ]:
events['result_name'].value_counts()

,count
result_name,
Successful,204422
Unsuccessful,42307
On Target,799
Blocked,733
Low Quality Shot,85
Off Target,61
Yellow_Card,29
Keeper Rush-Out,12


### LabelEncoder는 컬럼마다 독립적으로 만들어서 사용해야한다:


- 데이터 스케일링할때는 여러 칼럼에 같은 변환기를 공유해도 됨
- LabelEncoder는 각 컬럼마다 고유한 범주가 달라서 서로 다르게 사용해야함

In [ ]:
#카테고리 인코딩
events['type_name']=events['type_name'].fillna("__NA_TYPE__")
events['result_name']=events['result_name'].fillna("__NA_RES__")

le_type=LabelEncoder()
le_res=LabelEncoder()

events['type_id']=le_type.fit_transform(events['type_name']) #숫자로 변환
events['res_id']=le_res.fit_transform(events['result_name'])

#team_id(득점을 수행한 팀 ID)는 그대로 써도 되지만, 문자열이면 숫자로 매핑
if events['team_id'].dtype=='object':
  le_team=LabelEncoder()
  events['team_id_enc']=le_team.fit_transform(events['team_id'])
else:
  events['team_id_enc']=events['team_id'].astype(int)

In [ ]:
# 마지막 K 득점만 사용(lastK)

#rev_idx:0이 마지막 득점
events['rev_']